In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
property_sales = pd.read_csv('/Users/chintanjikkar/Desktop/PACE/Fall 25/Visual Analytics/Team Project/Processed data/nyc_sales_clean_2003_2025.csv')
crime_data = pd.read_csv('/Users/chintanjikkar/Desktop/PACE/Fall 25/Visual Analytics/Team Project/Processed data/Complaints_2015-2025.csv')
air_quality = pd.read_csv('/Users/chintanjikkar/Desktop/PACE/Fall 25/Visual Analytics/Team Project/Processed data/air_quality_processed_with_annual_2024_2025.csv')

/var/folders/26/k5qhk1wd7vld3l2ndf6041200000gn/T/ipykernel_2572/1504312379.py:1: DtypeWarning: Columns (3,7,9) have mixed types. Specify dtype option on import or set low_memory=False.
  property_sales = pd.read_csv('/Users/chintanjikkar/Desktop/PACE/Fall 25/Visual Analytics/Team Project/Processed data/nyc_sales_clean_2003_2025.csv')


In [3]:
print(property_sales.head())
print(crime_data.head())
print(air_quality.head())

   borough neighborhood building_class_category tax_class_at_present  block  \
0        2     BATHGATE    01  ONE FAMILY HOMES                    1   2917   
1        2     BATHGATE    01  ONE FAMILY HOMES                    1   3028   
2        2     BATHGATE    01  ONE FAMILY HOMES                    1   3030   
3        2     BATHGATE    01  ONE FAMILY HOMES                    1   3035   
4        2     BATHGATE    01  ONE FAMILY HOMES                    1   3035   

   lot  easement building_class_at_present                 address  \
0   15       NaN                        A1  1848 WASHINGTON AVENUE   
1   25       NaN                        A5     412 EAST 179 STREET   
2   55       NaN                        A5     410 EAST 182 STREET   
3    2       NaN                        S1     441 EAST 178 STREET   
4   52       NaN                        A1     461 EAST 178 STREET   

  apartment_number  ...  sale_month  sale_quarter  price_per_sqft  \
0              NaN  ...           5

In [4]:
print(property_sales.info())
print(crime_data.info())
print(air_quality.info())

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1294145 entries, 0 to 1294144
Data columns (total 34 columns):
 #   Column                          Non-Null Count    Dtype  
---  ------                          --------------    -----  
 0   borough                         1294145 non-null  int64  
 1   neighborhood                    1294145 non-null  object 
 2   building_class_category         1294050 non-null  object 
 3   tax_class_at_present            1129367 non-null  object 
 4   block                           1294145 non-null  int64  
 5   lot                             1294145 non-null  int64  
 6   easement                        0 non-null        float64
 7   building_class_at_present       1129367 non-null  object 
 8   address                         1294143 non-null  object 
 9   apartment_number                251037 non-null   object 
 10  zip_code                        1294100 non-null  float64
 11  residential_units               1181852 non-null  float64
 12  

In [5]:
print(property_sales.describe())
print(crime_data.describe())
print(air_quality.describe())

            borough         block           lot  easement      zip_code  \
count  1.294145e+06  1.294145e+06  1.294145e+06       0.0  1.294100e+06   
mean   2.928034e+00  4.180243e+03  3.290319e+02       NaN  1.082044e+04   
std    1.333100e+00  3.630694e+03  6.012908e+02       NaN  6.082453e+02   
min    1.000000e+00  1.000000e+00  1.000000e+00       NaN  0.000000e+00   
25%    1.000000e+00  1.268000e+03  2.000000e+01       NaN  1.028000e+04   
50%    3.000000e+00  3.116000e+03  4.600000e+01       NaN  1.120700e+04   
75%    4.000000e+00  6.212000e+03  2.020000e+02       NaN  1.135800e+04   
max    5.000000e+00  1.635000e+04  9.117000e+03       NaN  1.169700e+04   

       residential_units  commercial_units   total_units  land_square_feet  \
count       1.181852e+06      1.121330e+06  1.192145e+06      1.111037e+06   
mean        1.901702e+00      1.825859e-01  2.067332e+00      2.551874e+03   
std         2.712952e+01      7.062092e+00  2.794091e+01      3.562741e+04   
min         

In [6]:
from scipy.stats import mstats

# 2. Data Cleaning

# Clean Property Sales Data

# Drop irrelevant columns
property_sales = property_sales.drop(columns=['easement'])

# Convert sale_date to datetime
property_sales['sale_date'] = pd.to_datetime(property_sales['sale_date'], errors='coerce')

# Ensure consistent data types for 'borough' and 'neighborhood' columns
property_sales['borough'] = property_sales['borough'].astype(str).str.upper()

# Handle missing values for numerical columns by filling with mean or median
property_sales['price_per_sqft'].fillna(property_sales['price_per_sqft'].mean(), inplace=True)
property_sales['land_square_feet'].fillna(property_sales['land_square_feet'].median(), inplace=True)
property_sales['gross_square_feet'].fillna(property_sales['gross_square_feet'].median(), inplace=True)
property_sales['residential_units'].fillna(property_sales['residential_units'].median(), inplace=True)
property_sales['commercial_units'].fillna(property_sales['commercial_units'].median(), inplace=True)
property_sales['property_age'].fillna(property_sales['property_age'].median(), inplace=True)

# Handle outliers (winsorizing sale_price and price_per_sqft)
property_sales['sale_price'] = mstats.winsorize(property_sales['sale_price'], limits=[0.01, 0.01])
property_sales['price_per_sqft'] = mstats.winsorize(property_sales['price_per_sqft'], limits=[0.01, 0.01])

# Clean Crime Data

# Normalize 'BORO_NM' column
crime_data['BORO_NM'] = crime_data['BORO_NM'].str.upper()

# Replace zero latitude and longitude values with NaN, as they are likely errors
crime_data['Latitude'] = crime_data['Latitude'].replace(0, np.nan)
crime_data['Longitude'] = crime_data['Longitude'].replace(0, np.nan)

# Ensure 'Latitude' and 'Longitude' are in float64 format for geospatial analysis
crime_data['Latitude'] = crime_data['Latitude'].astype(float)
crime_data['Longitude'] = crime_data['Longitude'].astype(float)

# Handle missing crime data by filling with 'Unknown' or a reasonable value
crime_data['OFNS_DESC'].fillna('UNKNOWN', inplace=True)

# Clean Air Quality Data

# Ensure 'Geo Place Name' is consistent
air_quality['Geo Place Name'] = air_quality['Geo Place Name'].str.upper()

# Handle missing air quality values (if any) by filling with median or mean
air_quality['Data Value'].fillna(air_quality['Data Value'].median(), inplace=True)

/var/folders/26/k5qhk1wd7vld3l2ndf6041200000gn/T/ipykernel_2572/904994707.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  property_sales['price_per_sqft'].fillna(property_sales['price_per_sqft'].mean(), inplace=True)
/var/folders/26/k5qhk1wd7vld3l2ndf6041200000gn/T/ipykernel_2572/904994707.py:18: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediat

In [7]:
# Inspect column names in property_sales dataset to confirm the presence of 'easement'
print(property_sales.columns)

Index(['borough', 'neighborhood', 'building_class_category',
       'tax_class_at_present', 'block', 'lot', 'building_class_at_present',
       'address', 'apartment_number', 'zip_code', 'residential_units',
       'commercial_units', 'total_units', 'land_square_feet',
       'gross_square_feet', 'year_built', 'tax_class_at_time_of_sale',
       'building_class_at_time_of_sale', 'sale_price', 'sale_date',
       'source_file', 'borough_name', 'sale_year', 'sale_month',
       'sale_quarter', 'price_per_sqft', 'property_age', 'is_new_construction',
       'is_residential', 'has_ppsf', 'is_luxury_ppsf', 'is_extreme_price',
       'price_winsor'],
      dtype='object')


In [8]:
print("Property Sales Data (First 5 rows):")
print(property_sales.head())

print("Crime Data (First 5 rows):")
print(crime_data.head())

print("Air Quality Data (First 5 rows):")
print(air_quality.head())

# Check data types and for missing values
print("\nProperty Sales Data Info:")
print(property_sales.info())
print(property_sales.isnull().sum())

print("\nCrime Data Info:")
print(crime_data.info())
print(crime_data.isnull().sum())

print("\nAir Quality Data Info:")
print(air_quality.info())
print(air_quality.isnull().sum())

Property Sales Data (First 5 rows):
  borough neighborhood building_class_category tax_class_at_present  block  \
0       2     BATHGATE    01  ONE FAMILY HOMES                    1   2917   
1       2     BATHGATE    01  ONE FAMILY HOMES                    1   3028   
2       2     BATHGATE    01  ONE FAMILY HOMES                    1   3030   
3       2     BATHGATE    01  ONE FAMILY HOMES                    1   3035   
4       2     BATHGATE    01  ONE FAMILY HOMES                    1   3035   

   lot building_class_at_present                 address apartment_number  \
0   15                        A1  1848 WASHINGTON AVENUE              NaN   
1   25                        A5     412 EAST 179 STREET              NaN   
2   55                        A5     410 EAST 182 STREET              NaN   
3    2                        S1     441 EAST 178 STREET              NaN   
4   52                        A1     461 EAST 178 STREET              NaN   

   zip_code  ...  sale_month  sa

In [14]:
# Handle missing values without using inplace=True to avoid FutureWarning
property_sales['price_per_sqft'] = property_sales['price_per_sqft'].fillna(property_sales['price_per_sqft'].mean())
property_sales['land_square_feet'] = property_sales['land_square_feet'].fillna(property_sales['land_square_feet'].median())
property_sales['gross_square_feet'] = property_sales['gross_square_feet'].fillna(property_sales['gross_square_feet'].median())
property_sales['residential_units'] = property_sales['residential_units'].fillna(property_sales['residential_units'].median())
property_sales['commercial_units'] = property_sales['commercial_units'].fillna(property_sales['commercial_units'].median())
property_sales['property_age'] = property_sales['property_age'].fillna(property_sales['property_age'].median())

In [17]:
# Fill missing 'building_class_category' and 'building_class_at_present' with 'UNKNOWN'
property_sales['building_class_category'] = property_sales['building_class_category'].fillna('UNKNOWN')
property_sales['building_class_at_present'] = property_sales['building_class_at_present'].fillna('UNKNOWN')

In [18]:
# Check data types and for missing values
print("\nProperty Sales Data Info:")
print(property_sales.info())
print(property_sales.isnull().sum())

print("\nCrime Data Info:")
print(crime_data.info())
print(crime_data.isnull().sum())

print("\nAir Quality Data Info:")
print(air_quality.info())
print(air_quality.isnull().sum())


Property Sales Data Info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1294145 entries, 0 to 1294144
Data columns (total 33 columns):
 #   Column                          Non-Null Count    Dtype         
---  ------                          --------------    -----         
 0   borough                         1294145 non-null  object        
 1   neighborhood                    1294145 non-null  object        
 2   building_class_category         1294145 non-null  object        
 3   tax_class_at_present            1129367 non-null  object        
 4   block                           1294145 non-null  int64         
 5   lot                             1294145 non-null  int64         
 6   building_class_at_present       1294145 non-null  object        
 7   address                         1294143 non-null  object        
 8   apartment_number                251037 non-null   object        
 9   zip_code                        1294100 non-null  float64       
 10  residential_uni

In [19]:
# Drop rows where the 'address' column is missing (2 rows)
property_sales = property_sales.dropna(subset=['address'])

In [20]:
# Drop the specified columns from the dataset
columns_to_drop = ['apartment_number', 'tax_class_at_present', 'total_units', 'year_built']
property_sales = property_sales.drop(columns=columns_to_drop)

In [21]:
# Perform a column-wise sanity check for missing values across the dataset
missing_values_column_wise = property_sales.isnull().sum()

# Display the number of missing values for each column
print("Sanity Check - Missing Values Column-wise:")
print(missing_values_column_wise)

Sanity Check - Missing Values Column-wise:
borough                            0
neighborhood                       0
building_class_category            0
block                              0
lot                                0
building_class_at_present          0
address                            0
zip_code                          45
residential_units                  0
commercial_units                   0
land_square_feet                   0
gross_square_feet                  0
tax_class_at_time_of_sale          0
building_class_at_time_of_sale     0
sale_price                         0
sale_date                          0
source_file                        0
borough_name                       0
sale_year                          0
sale_month                         0
sale_quarter                       0
price_per_sqft                     0
property_age                       0
is_new_construction                0
is_residential                     0
has_ppsf                        

In [22]:
# First, let's check if we have zip codes missing in the dataset
missing_zip_codes = property_sales[property_sales['zip_code'].isnull()]

# Check the rows with missing zip codes
print(f"Rows with missing zip codes: {len(missing_zip_codes)}")

# Impute missing zip codes based on 'borough' or 'neighborhood'
# If the 'borough' or 'neighborhood' has a unique zip code, use that for filling

# Impute based on neighborhood
property_sales['zip_code'] = property_sales.groupby('neighborhood')['zip_code'].transform(lambda x: x.fillna(x.mode()[0]))

# If any zip codes are still missing, use 'borough' as fallback
property_sales['zip_code'] = property_sales.groupby('borough')['zip_code'].transform(lambda x: x.fillna(x.mode()[0]))

# Finally, if there are still any missing zip codes, impute with the most frequent zip code in the dataset
property_sales['zip_code'].fillna(property_sales['zip_code'].mode()[0], inplace=True)

# Check if there are any missing zip codes after imputation
missing_zip_codes_after = property_sales[property_sales['zip_code'].isnull()]

if len(missing_zip_codes_after) == 0:
    print("All missing zip codes have been successfully filled.")
else:
    print(f"Rows still with missing zip codes: {len(missing_zip_codes_after)}")

Rows with missing zip codes: 45
All missing zip codes have been successfully filled.


/var/folders/26/k5qhk1wd7vld3l2ndf6041200000gn/T/ipykernel_2572/1876308017.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  property_sales['zip_code'].fillna(property_sales['zip_code'].mode()[0], inplace=True)


In [23]:
# Perform a column-wise sanity check for missing values across the dataset
missing_values_column_wise = property_sales.isnull().sum()

# Display the number of missing values for each column
print("Sanity Check - Missing Values Column-wise:")
print(missing_values_column_wise)

Sanity Check - Missing Values Column-wise:
borough                           0
neighborhood                      0
building_class_category           0
block                             0
lot                               0
building_class_at_present         0
address                           0
zip_code                          0
residential_units                 0
commercial_units                  0
land_square_feet                  0
gross_square_feet                 0
tax_class_at_time_of_sale         0
building_class_at_time_of_sale    0
sale_price                        0
sale_date                         0
source_file                       0
borough_name                      0
sale_year                         0
sale_month                        0
sale_quarter                      0
price_per_sqft                    0
property_age                      0
is_new_construction               0
is_residential                    0
has_ppsf                          0
is_luxury_ppsf       

In [24]:
# Check data types and for missing values
print("\nProperty Sales Data Info:")
print(property_sales.info())
print(property_sales.isnull().sum())

print("\nCrime Data Info:")
print(crime_data.info())
print(crime_data.isnull().sum())

print("\nAir Quality Data Info:")
print(air_quality.info())
print(air_quality.isnull().sum())


Property Sales Data Info:
<class 'pandas.core.frame.DataFrame'>
Index: 1294143 entries, 0 to 1294144
Data columns (total 29 columns):
 #   Column                          Non-Null Count    Dtype         
---  ------                          --------------    -----         
 0   borough                         1294143 non-null  object        
 1   neighborhood                    1294143 non-null  object        
 2   building_class_category         1294143 non-null  object        
 3   block                           1294143 non-null  int64         
 4   lot                             1294143 non-null  int64         
 5   building_class_at_present       1294143 non-null  object        
 6   address                         1294143 non-null  object        
 7   zip_code                        1294143 non-null  float64       
 8   residential_units               1294143 non-null  float64       
 9   commercial_units                1294143 non-null  float64       
 10  land_square_feet    

In [25]:
# 1. Handle Missing Latitude and Longitude in Crime Data

# Check for rows with missing Latitude and Longitude
missing_lat_long = crime_data[crime_data[['Latitude', 'Longitude']].isnull().any(axis=1)]
print(f"Rows with missing Latitude or Longitude: {len(missing_lat_long)}")

# Impute missing Latitude and Longitude based on 'BORO_NM' (borough)
crime_data['Latitude'] = crime_data.groupby('BORO_NM')['Latitude'].transform(lambda x: x.fillna(x.mode()[0]))
crime_data['Longitude'] = crime_data.groupby('BORO_NM')['Longitude'].transform(lambda x: x.fillna(x.mode()[0]))

# If any Latitude and Longitude are still missing, try imputing based on 'LOC_OF_OCCUR_DESC' (location of occurrence)
crime_data['Latitude'] = crime_data.groupby('LOC_OF_OCCUR_DESC')['Latitude'].transform(lambda x: x.fillna(x.mode()[0]))
crime_data['Longitude'] = crime_data.groupby('LOC_OF_OCCUR_DESC')['Longitude'].transform(lambda x: x.fillna(x.mode()[0]))

# Final fallback: Impute remaining missing values in Latitude and Longitude with the most frequent value in the entire dataset
crime_data['Latitude'].fillna(crime_data['Latitude'].mode()[0], inplace=True)
crime_data['Longitude'].fillna(crime_data['Longitude'].mode()[0], inplace=True)

# Check if there are any missing Latitude and Longitude values after imputation
missing_lat_long_after = crime_data[crime_data[['Latitude', 'Longitude']].isnull().any(axis=1)]
if len(missing_lat_long_after) == 0:
    print("All missing Latitude and Longitude values have been successfully filled.")
else:
    print(f"Rows still with missing Latitude or Longitude: {len(missing_lat_long_after)}")

Rows with missing Latitude or Longitude: 19
All missing Latitude and Longitude values have been successfully filled.


/var/folders/26/k5qhk1wd7vld3l2ndf6041200000gn/T/ipykernel_2572/2458475901.py:16: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on which we are setting values always behaves as a copy.

For example, when doing 'df[col].method(value, inplace=True)', try using 'df.method({col: value}, inplace=True)' or df[col] = df[col].method(value) instead, to perform the operation inplace on the original object.


  crime_data['Latitude'].fillna(crime_data['Latitude'].mode()[0], inplace=True)
/var/folders/26/k5qhk1wd7vld3l2ndf6041200000gn/T/ipykernel_2572/2458475901.py:17: FutureWarning: A value is trying to be set on a copy of a DataFrame or Series through chained assignment using an inplace method.
The behavior will change in pandas 3.0. This inplace method will never work because the intermediate object on whi

In [26]:
# Check data types and for missing values
print("\nProperty Sales Data Info:")
print(property_sales.info())
print(property_sales.isnull().sum())

print("\nCrime Data Info:")
print(crime_data.info())
print(crime_data.isnull().sum())

print("\nAir Quality Data Info:")
print(air_quality.info())
print(air_quality.isnull().sum())


Property Sales Data Info:
<class 'pandas.core.frame.DataFrame'>
Index: 1294143 entries, 0 to 1294144
Data columns (total 29 columns):
 #   Column                          Non-Null Count    Dtype         
---  ------                          --------------    -----         
 0   borough                         1294143 non-null  object        
 1   neighborhood                    1294143 non-null  object        
 2   building_class_category         1294143 non-null  object        
 3   block                           1294143 non-null  int64         
 4   lot                             1294143 non-null  int64         
 5   building_class_at_present       1294143 non-null  object        
 6   address                         1294143 non-null  object        
 7   zip_code                        1294143 non-null  float64       
 8   residential_units               1294143 non-null  float64       
 9   commercial_units                1294143 non-null  float64       
 10  land_square_feet    

In [29]:
# 4. Save individual cleaned datasets
property_sales.to_csv('/Users/chintanjikkar/Desktop/PACE/Fall 25/Visual Analytics/Team Project/Cleaned_NYC_Property_Sales.csv', index=False)
crime_data.to_csv('/Users/chintanjikkar/Desktop/PACE/Fall 25/Visual Analytics/Team Project/Cleaned_NYPD_Complaints.csv', index=False)
air_quality.to_csv('/Users/chintanjikkar/Desktop/PACE/Fall 25/Visual Analytics/Team Project/Cleaned_NYC_Air_Quality.csv', index=False)

In [30]:
# 1. Merge Property Sales with Crime Data on 'borough' (for geographical consistency)
merged_data_with_crime = pd.merge(property_sales, crime_data, left_on='borough', right_on='BORO_NM', how='inner')

# 2. Merge the resulting dataset with Air Quality Data on 'borough' or 'Geo Place Name'
merged_data_final = pd.merge(merged_data_with_crime, air_quality, left_on='borough', right_on='Geo Place Name', how='inner')

# 3. Save the final merged dataset as a single CSV file
merged_data_final.to_csv('/Users/chintanjikkar/Desktop/PACE/Fall 25/Visual Analytics/Team Project/Final_Merged_Data.csv', index=False)